In [97]:
from google.cloud import bigquery

client = bigquery.Client(project="prod-organize-arizon-4e1c0a83")

query = """
SELECT
*
FROM `prod-organize-arizon-4e1c0a83.viewers_dataset.az_censustract_voters_2024`
"""


df = client.query(query).to_dataframe()

df.head()

,mailaddrcensustract20,geoid,population_count,dem_votes,rep_votes,dem_margin,third_votes
0,090202,530350902023013,171,35,35,1.000000,42
1,110002,040131100021004,6171,691,210,3.290476,423
2,None,06073,216,40,43,0.930233,41
3,020810,121090208101004,1120,156,355,0.439437,275
4,019900,360470199003001,137,50,8,6.250000,19


In [98]:
import geopandas as gpd

# so didn't know you can do this with raw url until today >:)
url = "https://raw.githubusercontent.com/cmarikos/az-landscape-2024/christina-dev/Analysis%20Files/az_census_tract.geojson"
gdf = gpd.read_file(url)


gdf.crs

<Geographic 2D CRS: EPSG:4269>
Name: NAD83
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: North America - onshore and offshore: Canada - Alberta; British Columbia; Manitoba; New Brunswick; Newfoundland and Labrador; Northwest Territories; Nova Scotia; Nunavut; Ontario; Prince Edward Island; Quebec; Saskatchewan; Yukon. Puerto Rico. United States (USA) - Alabama; Alaska; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Hawaii; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming. US Virgin Islands. British Virgin Islands

In [99]:
# above gdf.crs told me that we're in EPSG:4269
# folium uses EPSG:4326 so we gotta convert

gdf = gdf.to_crs(4326)

In [100]:
len(df["geoid"])

55349

In [101]:
df.head(10)

,mailaddrcensustract20,geoid,population_count,dem_votes,rep_votes,dem_margin,third_votes
0,090202,530350902023013,171,35,35,1.000000,42
1,110002,040131100021004,6171,691,210,3.290476,423
2,None,06073,216,40,43,0.930233,41
3,020810,121090208101004,1120,156,355,0.439437,275
4,019900,360470199003001,137,50,8,6.250000,19
5,010516,120210105161000,428,78,95,0.821053,85
6,002008,450190020083007,9899,1173,1302,0.900922,1062
7,004621,040190046211011,3564,715,636,1.124214,465
8,010704,010890107042014,2018,359,392,0.915816,450
9,072408,530530724081001,226,45,50,0.900000,54


In [102]:
import pandas as pd
import numpy as np

twoparty = df[["dem_votes", "rep_votes"]].sum(axis=1, skipna=True)
with np.errstate(divide='ignore', invalid='ignore'):
    df["dem_margin_2p"] = np.where(twoparty > 0,
                                    ( df["rep_votes"] - df["dem_votes"]) / twoparty,
                                    np.nan)

# this made my geoids not unique anymore
# no idea what the last four digits are here
# if the census bureau doesn't use them then eh
chop_four = df["tract_geoid11"] = (
    df["geoid"].astype("string")
      .str.replace(r"\D", "", regex=True)
      .str.zfill(15)
      .str[:11]
)

df.head(5)

,mailaddrcensustract20,geoid,population_count,dem_votes,rep_votes,dem_margin,third_votes,dem_margin_2p,tract_geoid11
0,090202,530350902023013,171,35,35,1.000000,42,0.000000,53035090202
1,110002,040131100021004,6171,691,210,3.290476,423,-0.533851,04013110002
2,None,06073,216,40,43,0.930233,41,0.036145,00000000000
3,020810,121090208101004,1120,156,355,0.439437,275,0.389432,12109020810
4,019900,360470199003001,137,50,8,6.250000,19,-0.724138,36047019900


In [103]:
import numpy as np
# Group by 'tract_geoid11' and sum the relevant columns
metrics = df.groupby("tract_geoid11").agg(
    population_count=("population_count", "sum"),
    dem_votes=("dem_votes", "sum"),
    rep_votes=("rep_votes", "sum"),
    third_votes=("third_votes", "sum")
).reset_index()

# Recalculate dem_margin_2p after aggregation
twoparty_grouped = metrics["dem_votes"] + metrics["rep_votes"]
with np.errstate(divide='ignore', invalid='ignore'):
    metrics["dem_margin_2p"] = np.where(twoparty_grouped > 0,
                                    (metrics["rep_votes"] - metrics["dem_votes"]) / twoparty_grouped,
                                    np.nan)

gdf = gdf.merge(metrics, left_on="GEOID", right_on="tract_geoid11", how="left")


In [104]:
import folium
import numpy as np
import pandas as pd
from branca.colormap import LinearColormap
from folium.features import GeoJson, GeoJsonTooltip


# ============================================
KEY_COL  = "GEOID"
POP_COL  = "population_count"
DEM_COL  = "dem_votes"
REP_COL  = "rep_votes"
OTH_COL  = "third_votes"
MARGIN_COL = "dem_margin_2p" # Changed from "dem_margin_2p_y"
# ============================================

# Filter gdf to include only Arizona (STATEFP '04')
gdf = gdf[gdf['STATEFP'] == '04']

minx, miny, maxx, maxy = gdf.total_bounds
m = folium.Map(location=[(miny+maxy)/2, (minx+maxx)/2], zoom_start=6, tiles="CartoDB Positron")

abs_margin = gdf[MARGIN_COL].abs().dropna() # Use MARGIN_COL
vmax = float(np.quantile(abs_margin, 0.98)) if len(abs_margin) else 1.0
cmap = LinearColormap(colors=["#2166ac", "#f7f7f7", "#b2182b"], vmin=-vmax, vmax=vmax)
cmap.caption = "Dem margin (two-party, Dem − Rep)"
cmap.add_to(m)


def style_fn(feat):
    val = feat["properties"].get(MARGIN_COL) # Use MARGIN_COL
    if pd.isna(val):
        return {"fillColor": "#cccccc", "fillOpacity": 0.25, "weight": 0.4, "color": "#666"}
    return {"fillColor": cmap(val), "fillOpacity": 0.8, "weight": 0.4, "color": "#666"}

tooltip_fields = [c for c in [KEY_COL, MARGIN_COL, POP_COL] if c in gdf.columns] # Use MARGIN_COL
tooltip_aliases = ["Pct:", "Dem margin (2-party):", "Population:"]

poly_layer = GeoJson(
    data=gdf.to_json(),
    name="Census tracts 2020 — Dem margin",
    style_function=style_fn,
    highlight_function=lambda f: {"weight": 2, "color": "#000"},
    tooltip=GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_aliases, localize=True, sticky=False),
)
poly_layer.add_to(m)

# Population “spikes” as scaled circle markers at representative points
# Population “spikes” as scaled circle markers at representative points
pop_layer = folium.FeatureGroup(name="Population spikes", show=False)

max_pop = gdf[POP_COL].max() if POP_COL in gdf.columns else None

def pop_radius(pop, min_r=3, max_r=18):
    if (pop is None) or pd.isna(pop) or (max_pop is None) or (max_pop <= 0):
        return 0
    return float(min_r + (max_r - min_r) * np.sqrt(pop / max_pop))

# --- SAFER: filter to valid geometries, compute points, drop null points & pops
gdf_pts = gdf[gdf.geometry.notna() & ~gdf.geometry.apply(lambda g: getattr(g, "is_empty", True))].copy()
gdf_pts["__pt"] = gdf_pts.geometry.apply(lambda g: g.representative_point() if (g is not None and not g.is_empty) else None)
gdf_pts = gdf_pts[gdf_pts["__pt"].notna() & gdf_pts[POP_COL].notna()].copy()

for _, row in gdf_pts.iterrows():
    r = pop_radius(row[POP_COL])
    if r <= 0:
        continue
    lat, lon = float(row["__pt"].y), float(row["__pt"].x)
    folium.CircleMarker(
        location=[lat, lon],
        radius=r,
        weight=0.8,
        color="#222",
        fill=True,
        fill_opacity=0.45,
        fill_color="#ffffff",
        popup=folium.Popup(
            f"Pct: {row.get(KEY_COL,'')}"
            + (f"<br>Population: {int(row[POP_COL]):,}" if pd.notna(row[POP_COL]) else "")
            + (f"<br>Dem margin (2p): {row[MARGIN_COL]:.3f}" if pd.notna(row.get(MARGIN_COL)) else ""), # Use MARGIN_COL
            max_width=320
        ),
    ).add_to(pop_layer)

pop_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m.save("az_precincts_demmargin_popspikes.html")

In [ ]:
from IPython.display import IFrame
IFrame("map.html", width="100%", height=600)

m